# Singular Value Decomposition — companion notebook

Runnable companion for [`svd.md`](./svd.md).

1. Decompose a small matrix by hand and via NumPy.
2. Show how the singular values explain matrix action geometrically (unit circle → ellipse).
3. Use Eckart–Young to compress an image via low-rank approximation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## 1. Small example matching the markdown sheet

$$ A = \begin{bmatrix} 3 & 0 \\ 4 & 5 \end{bmatrix} $$

In [ ]:
A = np.array([[3.0, 0.0],
              [4.0, 5.0]])
U, s, Vt = np.linalg.svd(A)
print('U =\n', U)
print('singular values =', s)
print('V^T =\n', Vt)

# Reconstruct
Sigma = np.diag(s)
print('\nU @ Sigma @ V^T =\n', U @ Sigma @ Vt)
print('\n||A - U Sigma V^T||_F =', np.linalg.norm(A - U @ Sigma @ Vt))

In [ ]:
# Cross-checks: ||A||_F, ||A||_2, condition number
print(f'Frobenius norm  : sqrt(sum s^2) = {np.sqrt((s**2).sum()):.4f}   np = {np.linalg.norm(A, "fro"):.4f}')
print(f'Spectral norm   : s[0]          = {s[0]:.4f}                np = {np.linalg.norm(A, 2):.4f}')
print(f'Condition num.  : s[0]/s[-1]    = {s[0]/s[-1]:.4f}                np = {np.linalg.cond(A):.4f}')

## 2. Geometric picture — unit circle becomes an ellipse

Apply $A$ to many unit vectors. The image is an ellipse whose semi-axis lengths are $\sigma_1, \sigma_2$ aligned with $U$.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 200)
circle = np.stack([np.cos(theta), np.sin(theta)], axis=0)  # (2, 200)
ellipse = A @ circle

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(circle[0], circle[1], 'tab:blue', label='unit circle (input)')
ax.plot(ellipse[0], ellipse[1], 'tab:orange', label='A * (unit circle)')

# Show the principal axes of the ellipse: U[:, i] * s[i]
for i in range(2):
    axis = U[:, i] * s[i]
    ax.plot([0, axis[0]], [0, axis[1]], color='tab:red', lw=2)
    ax.annotate(f'sigma_{i+1}={s[i]:.2f}', xy=axis * 1.05, color='tab:red')

ax.set_aspect('equal'); ax.legend(); ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_title('SVD geometry: A stretches the unit circle into an ellipse')
plt.show()

## 3. Low-rank approximation (Eckart–Young)

Keeping the top $k$ singular components gives the best rank-$k$ approximation in Frobenius norm. We demonstrate on a synthetic "image".

In [ ]:
# Build a synthetic image: a few low-frequency components + noise
n = 100
xs = np.linspace(0, 4 * np.pi, n)
img = (
    np.outer(np.sin(xs),   np.cos(xs))
    + 0.6 * np.outer(np.sin(2 * xs), np.cos(0.5 * xs))
    + 0.3 * np.outer(np.cos(0.3 * xs), np.sin(3 * xs))
)
img += 0.05 * rng.standard_normal(img.shape)

U, s, Vt = np.linalg.svd(img, full_matrices=False)

def rank_k(k):
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

ks = [1, 3, 5, 10, 30]
fig, axes = plt.subplots(1, len(ks) + 1, figsize=(3 * (len(ks) + 1), 3))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('original')
for ax, k in zip(axes[1:], ks):
    approx = rank_k(k)
    err = np.linalg.norm(img - approx) / np.linalg.norm(img)
    ax.imshow(approx, cmap='gray'); ax.set_title(f'k={k}\nrel err={err:.3f}')
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Singular value decay tells us the intrinsic dimensionality of the signal.
fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogy(s, marker='o', ms=3)
ax.set_xlabel('index i'); ax.set_ylabel('singular value sigma_i (log)')
ax.set_title('Singular value spectrum — fast decay means low effective rank')
ax.grid(True, which='both', alpha=0.3)
plt.show()

## 4. Pitfall demo — tiny singular values vs. true zeros

A theoretically rank-2 matrix, after a small random perturbation, has three nonzero singular values numerically. Use a tolerance, not exact zero.

In [ ]:
u1 = rng.standard_normal(5); u2 = rng.standard_normal(5)
v1 = rng.standard_normal(5); v2 = rng.standard_normal(5)
B = np.outer(u1, v1) + np.outer(u2, v2)
B += 1e-10 * rng.standard_normal(B.shape)
s = np.linalg.svd(B, compute_uv=False)
print('singular values:', s)

tol = max(B.shape) * s[0] * np.finfo(float).eps
print(f'\nnumpy.linalg.matrix_rank tolerance: {tol:.2e}')
print(f'numerical rank = {np.sum(s > tol)}  (true rank = 2)')